# MLFlow + BentoML End-to-End Model Lifecycle
## Training, Monitoring, and Deployment

## Abstract

This notebook demonstrates a **complete AI/ML pipeline** — from dataset preparation and model training to deployment with **BentoML** and **MLFlow** integration.  
It uses the **MedMNIST** dataset and a **CNN model**, running on **PC-AI** (Private Cloud AI) with MLFlow and S3 hosted in the platform.

### Pipeline Overview

| Step | Component | Tool |
|------|-----------|------|
| 1 | Data Prep | MedMNIST |
| 2 | Model Training | PyTorch + Accelerate |
| 3 | Experiment Tracking | MLFlow (PC-AI) |
| 4 | Model Packaging | BentoML |
| 5 | Deployment | MLIS / S3 |
| 6 | Inference | REST / FastAPI / Python |

---
## 1. Environment Setup

Before running any training or inference code, you need a dedicated Python environment with all the required packages.  
This step creates an isolated environment (`mlops-demo`) and registers it as a Jupyter kernel so the notebook always uses the correct dependencies.

### Step 1 — Create the environment and register the Jupyter kernel

Two options are provided below. **Run only the one that matches your setup**, then switch the kernel before continuing.

| Option | When to use |
|--------|-------------|
| **A — conda** | Recommended. Manages Python 3.11 automatically. |
| **B — venv** | Use if conda is not available; Python 3.11 must already be installed. |

In [1]:
ENV_NAME = "mlops-demo"


In [ ]:
# ── Option A: conda (recommended — manages Python version automatically) ──────
!conda create -n {ENV_NAME} python=3.11.11 -y -q
!conda run -n {ENV_NAME} pip install ipykernel
!conda run -n {ENV_NAME} python -m ipykernel install --user --name {ENV_NAME} --display-name "python({ENV_NAME})"
print(f"[conda] Kernel 'python({ENV_NAME})' registered.")

# # ── Option B: Python venv (Python must already be installed on the system) ─
# VENV_DIR = f".venv_{ENV_NAME}" # Will be used only if you are creating an virtual environment using venv
# !python -m venv {VENV_DIR}
# !{VENV_DIR}/bin/pip install ipykernel
# !{VENV_DIR}/bin/python -m ipykernel install --user --name {ENV_NAME} --display-name "python({ENV_NAME})"
# print(f"[venv] Kernel 'python({ENV_NAME})' registered. Venv location: {VENV_DIR}")

Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
doneecting package metadata (repodata.json): ...working... 
doneing environment: ...working... 

## Package Plan ##

  environment location: /home/test-guna-shared/.conda/envs/mlops-demo

  added / updated specs:
    - python=3.11.11


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_1          76 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-forge
    libgcc-15.2.0              |      he0feb66_19        1017 KB  

### ⚠️ **Switch the kernel now!**  
Select **python(mlops-demo)** from the kernel picker (top-right in VS Code / Jupyter) before running the next cell.  
This ensures the requirements are installed into the correct environment.  
**Note:** Sometimes in the kernel picker, the **python(mlops-demo)** kernel doesnot display immediately. In that case, Please refresh the browser. Then the **python(mlops-demo)** kernel will show up in the kernel picker.

In [1]:
# This ENV_NAME and the environment name in the first cell should be same. This is set once again here because 
# after switching to "python(mlops-demo)" kernel, this settings gets lost. Hence Setting it back
ENV_NAME = "mlops-demo" 
import sys
import os
# Get the active kernel name from the virtualenv/conda env path
current_kernel = os.path.basename(sys.prefix)

print(f"Active kernel environment: {current_kernel}")

if current_kernel != ENV_NAME:
    raise EnvironmentError(
        f"Wrong kernel! Expected '{ENV_NAME}' but got '{current_kernel}'.\n"
        f"Switch the kernel to 'python({ENV_NAME})' and re-run."
    )

print("Kernel check passed.")


Active kernel environment: mlops-demo
Kernel check passed.


### Step 1b — Ensure shell commands use the correct environment

When you run shell commands in a notebook cell (e.g. `!pip`, `!python`, `!uvicorn`), they inherit the system `PATH` rather than the active kernel's environment.  
The cell below prepends the kernel's `bin/` directory to `PATH`, so every `!` command in subsequent cells automatically resolves to the correct executables inside `mlops-demo`.

> This cell must be run **after** the kernel has been switched to `python(mlops-demo)`.

In [2]:
import sys
import os

# Get the bin/ directory of the active kernel's environment
kernel_bin = os.path.dirname(sys.executable)

# Prepend it to PATH so shell commands (!pip, !uvicorn, !python) use the kernel's env
os.environ["PATH"] = kernel_bin + os.pathsep + os.environ["PATH"]

# Verify
print("Kernel Python :", sys.executable)
print("PATH (first)  :", os.environ["PATH"].split(os.pathsep)[0])

Kernel Python : /home/test-guna-shared/.conda/envs/mlops-demo/bin/python
PATH (first)  : /home/test-guna-shared/.conda/envs/mlops-demo/bin


### Step 2 — Install project dependencies

With the kernel switched to `python(mlops-demo)` and `PATH` pointing at the right environment, run the cell below to install all packages listed in `code/requirements.txt`.

> **Important:** Do not run this cell with the default kernel — packages would be installed into the wrong Python environment and won't be available during training or inference.

In [3]:
# Run this cell AFTER switching the kernel to python(mlops-demo)
# pip will install into whichever kernel/environment is currently active. 
# Install it via the kernel's own executable using the sys.executable otherwise it will install in different python
# environment.

!python -m pip install -r code/requirements.txt --quiet
print("Dependencies installed into the active kernel environment.")

Dependencies installed into the active kernel environment.


---
## 2. Configuration — Environment Variables

Set the PC-AI MLFlow tracking URI and S3 endpoint URL before training.

> **Where to find these values in PC-AI:**  
> Navigate to **Tools & Frameworks → MLFlow → Configurations** and look for `extraEnvVars`:  
> - `MLFLOW_TRACKING_URI`  
> - `MLFLOW_S3_ENDPOINT_URL`

For Shared Project, 
> - Please refer to https://github.com/ai-solution-eng/ce-internal/blob/gunalan/shared_project/Authentication%26APIAccessGuide-SharedProject.ipynb for fetching the client secret and refresh token for built in UA Client in Keycloak.
> - Please refer to the procedure listed in https://github.com/ai-solution-eng/ce-internal/blob/gunalan/shared_project/MLflow_Keycloak_Client_Request.docx for creating a new client in keycloak. 


In [4]:
!cat code/.env.pcai

MLFLOW_TRACKING_URI="http://mlflow.mlflow.svc.cluster.local:5000"
MLFLOW_S3_ENDPOINT_URL="http://local-s3-service.ezdata-system.svc.cluster.local:30000"

## -------------------------------------------------------------
# Shared Project related environment settings. 
# Note: Uncomment this ONLY if you are running in shared project
## -------------------------------------------------------------
# Set the below environment variable only when you are running in shared project.
KEYCLOAK_TOKEN_URL="https://keycloak.hpepcai3.demo.local/realms/UA/protocol/openid-connect/token"

# In shared project, if you are using the built-in "ua" client in keycloak, uncomment the below
KEYCLOAK_CLIENT_ID="ua"
KEYCLOAK_CLIENT_SECRET="t2UVsCIwORkUBB7pr0IplAXI"
KEYCLOAK_VERIFY_TLS="False"
KEYCLOAK_UA_REFRESH_TOKEN="eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICI0NGQwMWFhZC1mZmI5LTQ3NTMtYjA5ZS0zZmE1NWJkZWJiYWUifQ.eyJpYXQiOjE3ODE4OTI0ODMsImp0aSI6IjEyYjM1YTQ0LTRkZGQtNGIwZC04MGNkLTg1MGM1MmY0YmNkZSIsImlzcyI6Im

In [5]:
import os
from dotenv import load_dotenv

load_dotenv("code/.env.pcai", override=True)

# Verify
print("MLFLOW_TRACKING_URI:", os.getenv("MLFLOW_TRACKING_URI"))
print("MLFLOW_S3_ENDPOINT_URL:", os.getenv("MLFLOW_S3_ENDPOINT_URL"))


MLFLOW_TRACKING_URI: http://mlflow.mlflow.svc.cluster.local:5000
MLFLOW_S3_ENDPOINT_URL: http://local-s3-service.ezdata-system.svc.cluster.local:30000


### Step 3 — Change working directory to `code/`

All scripts (`train_multi_gpu.py`, `infer.py`, etc.) and relative paths used throughout this notebook assume that `code/` is the working directory.  
Run the cell below **once** to move into it. Subsequent `!python` calls will resolve files relative to this directory.

In [6]:
%cd code


/home/test-guna-shared/ce-internal/mlflow-bentoml-end-to-end-demo/code


---
## 3. Data Preprocessing and Preparation

Download the **MedMNIST** (PathMNIST) dataset splits needed for training and inference.  
The `download_medmnist.py` script fetches samples from the specified split and saves them under `code/datasets/`.

Adjust `--num-samples` to download more data for a full training run (20 is used here as a quick test).


In [7]:
# Download training and test samples for PathMNIST
!python download_medmnist.py --data-split train --num-samples 20
!python download_medmnist.py --data-split test --num-samples 20


INFO:root:Downloading dataset: pathmnist
INFO:root:Downloading dataset: pathmnist


---
## 4. Model Training

Train a CNN model on the **MedMNIST** (PathMNIST) dataset with multi-GPU support via the Accelerate library.  
MLFlow tracks all metrics and artifacts during training. Monitor progress in the MLFlow application under **Tools & Frameworks → Mlflow**.

The example below runs a quick test with `--epochs 1` and `--num-samples 20`. For a full training run, remove or increase these arguments.  
See [`code/train_multi_gpu.py`](code/train_multi_gpu.py) for all configurable arguments (lr, epochs, batch size, etc.).  
**Note: Use --sharedproject if you are operating within the shared environment**

In [10]:
#!python train_multi_gpu.py --epochs 1 --experiment-name "mlflow_test-1" --num-samples 20 

# For shared project, pass the --sharedproject as  part of the commandline
!python train_multi_gpu.py --epochs 1 --experiment-name "mlflow_test-1" --num-samples 20 --sharedproject


INFO:root:Using MLflow tracking URI: http://mlflow.mlflow.svc.cluster.local:5000
INFO:root:Using MLFLOW_S3_ENDPOINT_URL: http://local-s3-service.ezdata-system.svc.cluster.local:30000
Authenticating with Built-in UA Client
/home/test-guna-shared/.conda/envs/mlops-demo/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak.hpepcai3.demo.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
INFO:root:MLFlow Token refreshed
INFO:root:eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJRZlVjVHlSY1BkUHBDdW90Y1JTc2F0M0twRzFHQW1XcjNDLWY2MWRuYUY0In0.eyJleHAiOjE4MTI3MTkwMTMsImlhdCI6MTc4MTk2MDYxMywiYXV0aF90aW1lIjoxNzgxOTYwNTYwLCJqdGkiOiJjYzYyMmI5Ni0yMzA3LTRkMTEtYTJiZS1hMWQ0NGJmYTE3NGUiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLmhwZXBjYWkzLmRlbW8ubG9jYWwvcmVhbG1zL1VBIiwic3ViIjoiNzMzYWViZDQtMDRkNC00YWI4LTk1YmMtOTI0MWY5ZGQ3

> **Monitor training in real time:** Open **Tools & Frameworks → AIE → MLFlow** in PC-AI.  
> Each run is logged under the experiment name you specified (e.g. `mlflow_test-1`).  
> You can inspect loss curves, accuracy metrics, and saved artifacts as training progresses.

---
## 5. Best Model Selection

After training completes, `pick_best.py` queries MLFlow for the run with the best validation metric within the specified experiment, then exports that model as a BentoML artifact.  

Update the `--experiment` name below to match the one used during training.

In [11]:
#!python pick_best.py --experiment "mlflow_test-1"

# For shared project, pass the --sharedproject as  part of the commandline
!python pick_best.py --experiment "mlflow_test-1" --sharedproject

INFO:root:Using MLflow tracking URI: http://mlflow.mlflow.svc.cluster.local:5000
INFO:root:Using MLFLOW_S3_ENDPOINT_URL: http://local-s3-service.ezdata-system.svc.cluster.local:30000
Authenticating with Built-in UA Client
/home/test-guna-shared/.conda/envs/mlops-demo/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak.hpepcai3.demo.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
INFO:root:MLFlow Token refreshed
INFO:root:eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJRZlVjVHlSY1BkUHBDdW90Y1JTc2F0M0twRzFHQW1XcjNDLWY2MWRuYUY0In0.eyJleHAiOjE4MTI3MTkxMjgsImlhdCI6MTc4MTk2MDcyOCwiYXV0aF90aW1lIjoxNzgxOTYwNTYwLCJqdGkiOiI5MzZkMGVhZi1iYmM4LTQ5YzYtOTI0My0wYjRiYjVjYTBlNDYiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLmhwZXBjYWkzLmRlbW8ubG9jYWwvcmVhbG1zL1VBIiwic3ViIjoiNzMzYWViZDQtMDRkNC00YWI4LTk1YmMtOTI0MWY5ZGQ3

In [12]:
# ── After running pick_best.py, set the values printed in the output ─────────
# Example output:
#   INFO:root:Best run: run_id: 90e141f06f3f472d854dd9b8f7393eaf - metric: 0.8857
#   INFO:root:Model URI to load: >> models:/medmnist_cnn/3
#   INFO:root:Model exported with BentoML: medmnist_cnn:e4jgrjv6not6rvtm

import os

MODEL_URI = "models:/medmnist_cnn/15"          # <-- update with your model URI
BENTO_MODEL_ARTIFACT = "medmnist_cnn:xsvtzadmvc6zwutg"  # <-- update with your tag

os.environ["MODEL_URI"] = MODEL_URI
os.environ["BENTO_MODEL_ARTIFACT"] = BENTO_MODEL_ARTIFACT

print(f"MODEL_URI             : {MODEL_URI}")
print(f"BENTO_MODEL_ARTIFACT  : {BENTO_MODEL_ARTIFACT}")

MODEL_URI             : models:/medmnist_cnn/15
BENTO_MODEL_ARTIFACT  : medmnist_cnn:xsvtzadmvc6zwutg


---
## 6. Inference

Before deploying to MLIS, verify that the trained model produces correct predictions locally.  
Three inference methods are available — pick the one that best suits your workflow:

| Option | Method | When to use |
|--------|--------|-------------|
| **6a** | Python script — local file | Quick single-image check from disk |
| **6b** | Python script — dataset file ID | Test against a specific sample from the downloaded dataset |
| **6c** | FastAPI server | Interactive testing via HTTP; mirrors how the model will be called in production |

### 6a. Option 1 — Inference from a Local File

Runs `infer.py` with a file path argument. The script loads the model from the MLFlow model registry using `MODEL_URI`, preprocesses the image, runs the CNN, and prints the predicted class.

The test dataset was downloaded in **Step 3**, so there is already a set of sample images under `datasets/medmnist/pathmnist/`. Update `FILE_PATH` to point to a different image if needed.

In [24]:
# TODO: Modify the infer.py code to get it as a command line
os.environ["SHARED_PROJECT"] = "True"
import os

MODEL_URI = os.environ.get("MODEL_URI")
FILE_PATH = "./datasets/medmnist/pathmnist/test/000000.png"  # update path as needed

!python infer.py --model-uri {MODEL_URI} --file-path {FILE_PATH}


Authenticating with Built-in UA Client
/home/test-guna-shared/.conda/envs/mlops-demo/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak.hpepcai3.demo.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
INFO:root:MLFlow Token refreshed
INFO:root:eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJRZlVjVHlSY1BkUHBDdW90Y1JTc2F0M0twRzFHQW1XcjNDLWY2MWRuYUY0In0.eyJleHAiOjE4MTI3MjA4MDEsImlhdCI6MTc4MTk2MjQwMSwiYXV0aF90aW1lIjoxNzgxOTYwNTYwLCJqdGkiOiI4MGQ4OGNlOS05NzI3LTQ1MTMtOTY5Mi1mY2YwMjc0YTNiYTUiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLmhwZXBjYWkzLmRlbW8ubG9jYWwvcmVhbG1zL1VBIiwic3ViIjoiNzMzYWViZDQtMDRkNC00YWI4LTk1YmMtOTI0MWY5ZGQ3Yjg0IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidWEiLCJub25jZSI6IjNMQ3pRemlwY0FPVVlpcmxJU3lXTDFna2pDb1U2MzhBNk9jbWNqQzMwc2siLCJzZXNzaW9uX3N0YXRlIjoiMTAwYzhiZjAtOGNhZC00MWM2LWI3MDQtNTllNjE1MDBlZjg

### 6b. Option 2 — Inference from a Dataset File ID

Instead of providing a file path, you can reference a sample directly by its zero-based index in the MedMNIST test dataset.  
`infer.py` will look up the image internally using the dataset loader. Set `FILE_ID` to the index of the sample you want to test.

In [25]:
# TODO: Modify the infer.py code to get it as a command line
os.environ["SHARED_PROJECT"] = "True"

import os

MODEL_URI = os.environ.get("MODEL_URI")
FILE_ID = "0"  # index within the MedMNIST test dataset

!python infer.py --model-uri {MODEL_URI} --file-id {FILE_ID}


Authenticating with Built-in UA Client
/home/test-guna-shared/.conda/envs/mlops-demo/lib/python3.11/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'keycloak.hpepcai3.demo.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
INFO:root:MLFlow Token refreshed
INFO:root:eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJRZlVjVHlSY1BkUHBDdW90Y1JTc2F0M0twRzFHQW1XcjNDLWY2MWRuYUY0In0.eyJleHAiOjE4MTI3MjA4NzIsImlhdCI6MTc4MTk2MjQ3MiwiYXV0aF90aW1lIjoxNzgxOTYwNTYwLCJqdGkiOiI1MTcyZDg4ZS05YzU3LTQyYjktOWVkYi0xYTZmNmU3ZmExODciLCJpc3MiOiJodHRwczovL2tleWNsb2FrLmhwZXBjYWkzLmRlbW8ubG9jYWwvcmVhbG1zL1VBIiwic3ViIjoiNzMzYWViZDQtMDRkNC00YWI4LTk1YmMtOTI0MWY5ZGQ3Yjg0IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidWEiLCJub25jZSI6IjNMQ3pRemlwY0FPVVlpcmxJU3lXTDFna2pDb1U2MzhBNk9jbWNqQzMwc2siLCJzZXNzaW9uX3N0YXRlIjoiMTAwYzhiZjAtOGNhZC00MWM2LWI3MDQtNTllNjE1MDBlZjg

### 6c. Option 3 — FastAPI Inference Server

Launch the Uvicorn server as a **background process** from within the notebook using `subprocess.Popen`.  
This returns immediately so you can continue to the next cells and call the endpoints while the server is running.  
Remember to run the shutdown cell when you are done to free the port.

In [ ]:
import subprocess, time, os
import sys

# Launch uvicorn in the background — Popen returns immediately
fastapi_proc = subprocess.Popen(
    ["uvicorn", "infer:app", "--host", "127.0.0.1", "--port", "8015"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait briefly for the server to finish starting up
time.sleep(10)

if fastapi_proc.poll() is None:
    print(f"FastAPI server is running in the background (PID: {fastapi_proc.pid})")
    print("Base URL: http://127.0.0.1:8015")
    print("Docs:     http://127.0.0.1:8015/docs")
else:
    stdout, stderr = fastapi_proc.communicate()
    print("Server failed to start:")
    print(stderr.decode())
#-----------------------------------------------------------------------------------
# Please run this in a seperate terminal.
# Equivalent terminal command:
#  export SHARED_PROJECT="True" (True/False based on the project)
#  uvicorn infer:app --host 127.0.0.1 --port 8015 --reload


In [26]:
# Option A — Predict from a local file
import requests

file_path = "datasets/medmnist/pathmnist/test/000000.png"  # update as needed

with open(file_path, "rb") as f:
    response = requests.post(
        "http://127.0.0.1:8015/predict_from_filepath",
        files={"file": f}
    )
print(response.json())

{'predicted_label': 5, 'probabilities': [0.10134190320968628, 0.11690343171358109, 0.10829047113656998, 0.11155547201633453, 0.11543749272823334, 0.127459317445755, 0.10693799704313278, 0.0979464054107666, 0.114127516746521]}


In [27]:
# Option B — Predict from a dataset file ID
import requests

response = requests.post(
    "http://127.0.0.1:8015/predict_from_file_id",
    data={"file_id": "0"}
)
print(response.json())


{'predicted_label': 5, 'probabilities': [0.10134190320968628, 0.11690343171358109, 0.10829047113656998, 0.11155547201633453, 0.11543749272823334, 0.127459317445755, 0.10693799704313278, 0.0979464054107666, 0.114127516746521], 'target_label': 8}


In [ ]:
# Shut down the FastAPI server when done
fastapi_proc.terminate()
fastapi_proc.wait()
print(f"FastAPI server (PID: {fastapi_proc.pid}) stopped.")

---
## 7. BentoML Packaging and Deployment

This section packages the trained model into a **Bento archive** — a self-contained bundle that includes the model weights, serving code, dependencies, and metadata.  
The archive is what gets uploaded to S3 and deployed to MLIS in the following steps.

The workflow here is:
1. **Verify** the saved BentoML model artifact
2. **Test locally** with `bentoml serve` before building the final archive
3. **Build** the Bento package
4. **Fix the archive** to meet MLIS requirements

### 7a. Verify the BentoML Model Artifact

Confirm that `pick_best.py` successfully exported the model to BentoML's local store.  
The tag printed here (e.g. `medmnist_cnn:e4jgrjv6not6rvtm`) is what you'll use in the cells below.

In [28]:
# List all saved BentoML model artifacts
!bentoml models list

 Tag                            Module          Size      Creation Time       
 medmnist_cnn:xsvtzadmvc6zwutg  bentoml.mlflow  1.50 MiB  2026-06-20 13:05:40 
 medmnist_cnn:pjkcootlds6pyutg  bentoml.mlflow  1.50 MiB  2026-06-18 13:49:08 
 medmnist_cnn:4zw6gxdix64cuutg  bentoml.mlflow  1.50 MiB  2026-06-15 13:41:23 
 medmnist_cnn:loyhyidgrkl7gutg  bentoml.mlflow  1.50 MiB  2026-06-12 18:13:05 


In [29]:
import os

# ── Update with the tag printed by pick_best.py or bentoml models list ───────
BENTO_MODEL_ARTIFACT = os.environ.get("BENTO_MODEL_ARTIFACT", "medmnist_cnn:xsvtzadmvc6zwutg")
os.environ["BENTO_MODEL_ARTIFACT"] = BENTO_MODEL_ARTIFACT

print(f"BENTO_MODEL_ARTIFACT: {BENTO_MODEL_ARTIFACT}")

BENTO_MODEL_ARTIFACT: medmnist_cnn:xsvtzadmvc6zwutg


### 7b. Serve the Bento Locally (Test)

Launch `bentoml serve` as a **background process** using `subprocess.Popen`.  
BentoML loads the model during startup, so a slightly longer wait (5 s) is used before checking the server is ready.  
Call the endpoint in the next cell, then run the shutdown cell when done.

In [ ]:
import subprocess, time
import sys

# Launch bentoml serve in the background — Popen returns immediately
bento_proc = subprocess.Popen(
    ["bentoml", "serve", "service:MedmnistCNNModel", "--working-dir", "deploy/v_latest/"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to finish starting up (BentoML typically takes a few seconds)
time.sleep(10)

if bento_proc.poll() is None:
    print(f"BentoML server is running in the background (PID: {bento_proc.pid})")
    print("Base URL: http://localhost:3000")
    print("Docs:     http://localhost:3000/docs")
else:
    stdout, stderr = bento_proc.communicate()
    print("Server failed to start:")
    print(stderr.decode())
# ------------------------------------------------------------------------------------------------ 
# Please run the below command in a seperate terminal if the above method is not working
# Equivalent terminal command: 
    # export BENTO_MODEL_ARTIFACT="medmnist_cnn:tiygyntmrc4w46wx"
    # bentoml serve service:MedmnistCNNModel --working-dir deploy/v_latest/

In [30]:
# Invoke the BentoML endpoint (server must be running from the cell above)
import requests

url = "http://localhost:3000/predict"
image_link = "https://drive.google.com/uc?export=download&id=1-jQJ6kvDNEtK_CZaRFicvMDWYz7_qfaf"

response = requests.post(url, json={"input_data": image_link})
print(response.json())

{'input_url': 'https://drive.google.com/uc?export=download&id=1-jQJ6kvDNEtK_CZaRFicvMDWYz7_qfaf', 'prediction': 2, 'probabilities': [0.10967858135700226, 0.11114255338907242, 0.11894809454679489, 0.10940832644701004, 0.11263544857501984, 0.1169632151722908, 0.10220064967870712, 0.1135796457529068, 0.10544347018003464]}


In [ ]:
# Shut down the BentoML server when done
bento_proc.terminate()
bento_proc.wait()
print(f"BentoML server (PID: {bento_proc.pid}) stopped.")

### 7c. Build the Bento Package

This step packages everything — model weights, the service definition, dependencies, and metadata — into a versioned Bento archive.  
The build reads `bentofile.yaml` in `deploy/v_latest/`, which specifies what to include.

The cell below first navigates into `deploy/v_latest/` (where `bentofile.yaml` lives), then runs `bentoml build`.  
**Note the tag printed at the end** (e.g. `medmnist_cnn_model:ubj6spf5dw2gdvtm`) — you will need it in the next step.

In [31]:
%cd deploy/v_latest

/home/test-guna-shared/ce-internal/mlflow-bentoml-end-to-end-demo/code/deploy/v_latest


In [32]:
!bentoml build
# Example output:
#   Successfully built Bento(tag="medmnist_cnn_model:ubj6spf5dw2gdvtm")

INFO: Adding BentoML requirement to the image: bentoml==1.4.39.
INFO: Locking PyPI package versions.

██████╗ ███████╗███╗   ██╗████████╗ ██████╗ ███╗   ███╗██╗
██╔══██╗██╔════╝████╗  ██║╚══██╔══╝██╔═══██╗████╗ ████║██║
██████╔╝█████╗  ██╔██╗ ██║   ██║   ██║   ██║██╔████╔██║██║
██╔══██╗██╔══╝  ██║╚██╗██║   ██║   ██║   ██║██║╚██╔╝██║██║
██████╔╝███████╗██║ ╚████║   ██║   ╚██████╔╝██║ ╚═╝ ██║███████╗
╚═════╝ ╚══════╝╚═╝  ╚═══╝   ╚═╝    ╚═════╝ ╚═╝     ╚═╝╚══════╝

Successfully built Bento(tag="medmnist_cnn_model:qx5mnytmvwkjeutg").

Next steps: 

* Deploy to BentoCloud:
    $ bentoml deploy medmnist_cnn_model:qx5mnytmvwkjeutg -n ${DEPLOYMENT_NAME}

* Update an existing deployment on BentoCloud:
    $ bentoml deployment update --bento medmnist_cnn_model:qx5mnytmvwkjeutg 
${DEPLOYMENT_NAME}

* Containerize your Bento with `bentoml containerize`:
    $ bentoml containerize medmnist_cnn_model:qx5mnytmvwkjeutg 

* Push to BentoCloud with `bentoml push`:
    $ bentoml push medmnist_cnn_model:q

### 7d. Fix Bento Archive for MLIS

MLIS deployment expects an `install.sh` file (generated by BentoML v1.1.11 during build).  
The `fix_bento_archive.sh` script adds this file and creates a `.bento` archive.

> See [`code/deploy/v_latest/install.sh`](code/deploy/v_latest/install.sh) and [`code/deploy/v_latest/requirements.txt`](code/deploy/v_latest/requirements.txt) for the files that are injected.

In [33]:
# Update with the Bento tag from the build step above
BENTO_TAG = "medmnist_cnn_model:qx5mnytmvwkjeutg"  # <-- update

!sh fix_bento_archive.sh {BENTO_TAG}
# Output: medmnist_cnn_demo-bento-v-latest.bento created in code/deploy/v_latest/

mkdir: cannot create directory ‘/home/test-guna-shared/ce-internal/mlflow-bentoml-end-to-end-demo/code/deploy/v_latest/bento_inspect_folder’: File exists
Bento(tag="medmnist_cnn_model:qx5mnytmvwkjeutg") exported to 
/home/test-guna-shared/ce-internal/mlflow-bentoml-end-to-end-demo/code/deploy/v_
latest/bento_inspect_folder/tmp_bento.bento.
./
./README.md
./apis/
./apis/openapi.yaml
./apis/schema.json
./bento.yaml
./env/
./env/docker/
./env/docker/Dockerfile
./env/docker/entrypoint.sh
./env/python/
./env/python/requirements.in
./env/python/requirements.txt
./models/
./models/medmnist_cnn/
./models/medmnist_cnn/latest
./models/medmnist_cnn/xsvtzadmvc6zwutg/
./models/medmnist_cnn/xsvtzadmvc6zwutg/mlflow_model/
./models/medmnist_cnn/xsvtzadmvc6zwutg/mlflow_model/MLmodel
./models/medmnist_cnn/xsvtzadmvc6zwutg/mlflow_model/conda.yaml
./models/medmnist_cnn/xsvtzadmvc6zwutg/mlflow_model/data/
./models/medmnist_cnn/xsvtzadmvc6zwutg/mlflow_model/data/model.pth
./models/medmnist_cnn/xsvtzadmvc6zw

---
## 8. Upload Model to PC-AI Local S3

Uses the **local S3 service** built into PC-AI — no external credentials required.

- **Endpoint:** `http://local-s3-service.ezdata-system.svc.cluster.local:30000`
- **Bucket:** set `S3_BUCKET_NAME` below (create it in the next cell if it doesn't exist yet)

In [34]:
import boto3
from botocore.exceptions import ClientError

# ── Configuration ─────────────────────────────────────────────────────────────
S3_BUCKET_NAME = "mlops-demo"   # <-- update if needed
S3_ENDPOINT    = "http://local-s3-service.ezdata-system.svc.cluster.local:30000"

# # Initialize the S3 client (no credentials needed for the local PC-AI S3 service)
# s3_client = boto3.client("s3", endpoint_url=S3_ENDPOINT)

# print("Existing buckets:")
# print(s3_client.list_buckets()["Buckets"])

# # Create the bucket only if it does not exist (404); re-raise any other errors
# try:
#     s3_client.head_bucket(Bucket=S3_BUCKET_NAME)
#     print(f"Bucket '{S3_BUCKET_NAME}' already exists — skipping creation.")
# except ClientError as e:
#     error_code = int(e.response["Error"]["Code"])
#     if error_code == 404:
#         s3_client.create_bucket(Bucket=S3_BUCKET_NAME)
#         print(f"Bucket '{S3_BUCKET_NAME}' created successfully.")
#     else:
#         raise 

In [35]:
%cd ../../

/home/test-guna-shared/ce-internal/mlflow-bentoml-end-to-end-demo/code


In [38]:
BENTO_FILE = "deploy/v_latest/medmnist_cnn_demo-bento-v-latest.bento"

# !python upload_model_to_s3_bucket.py \
#     --s3-bucket-name {S3_BUCKET_NAME} \
#     --endpoint-url {S3_ENDPOINT} \
#     --bento-file {BENTO_FILE} 

!python upload_model_to_s3_bucket.py \
    --s3-bucket-name {S3_BUCKET_NAME} \
    --endpoint-url {S3_ENDPOINT} \
    --bento-file {BENTO_FILE} \
    --shared

Authenticating with Built-in UA Client
INFO:root:MLFlow Token refreshed
INFO:root:eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJRZlVjVHlSY1BkUHBDdW90Y1JTc2F0M0twRzFHQW1XcjNDLWY2MWRuYUY0In0.eyJleHAiOjE4MTI3MjE0NzksImlhdCI6MTc4MTk2MzA3OSwiYXV0aF90aW1lIjoxNzgxOTYwNTYwLCJqdGkiOiI4OWQxNjNlOS1hOWQxLTQyZmUtODBkMi0wY2M2MGRjNTg3YzkiLCJpc3MiOiJodHRwczovL2tleWNsb2FrLmhwZXBjYWkzLmRlbW8ubG9jYWwvcmVhbG1zL1VBIiwic3ViIjoiNzMzYWViZDQtMDRkNC00YWI4LTk1YmMtOTI0MWY5ZGQ3Yjg0IiwidHlwIjoiQmVhcmVyIiwiYXpwIjoidWEiLCJub25jZSI6IjNMQ3pRemlwY0FPVVlpcmxJU3lXTDFna2pDb1U2MzhBNk9jbWNqQzMwc2siLCJzZXNzaW9uX3N0YXRlIjoiMTAwYzhiZjAtOGNhZC00MWM2LWI3MDQtNTllNjE1MDBlZjg2IiwiYWNyIjoiMSIsInNjb3BlIjoib3BlbmlkIG9mZmxpbmVfYWNjZXNzIGVtYWlsIHByb2ZpbGUgZGRwY2FpaDIwMDp1YSIsInNpZCI6IjEwMGM4YmYwLThjYWQtNDFjNi1iNzA0LTU5ZTYxNTAwZWY4NiIsInVpZCI6IjEwMDAwMTE4IiwiZ2lkIjoiMTAwMSIsImVtYWlsX3ZlcmlmaWVkIjpmYWxzZSwibmFtZXNwYWNlIjoicHJvamVjdC11c2VyLWd1bmFsYW5wdiIsIm5hbWUiOiJHdW5hbGFuIFBlcnVtYWwgVmlqYXlhbiIsImdyb3VwcyI6WyJ1YS1lbmFibGVkIiwib2ZmbG

---
## 9. Deploy in MLIS

1. **Launch MLIS** → under *Tools & Frameworks*
2. **Create Registry** (type S3) pointing to the local PC-AI S3 service
3. **Package Model** with the settings below
4. **Deploy** with `fixed-1` scaling
5. **Generate Access Token**

### Registry Settings
```
Name:        e.g. local-s3-registry
Type:        S3
Bucket name: mlops-demo               (same as S3_BUCKET_NAME used in Step 8)
Endpoint:    http://local-s3-service.ezdata-system.svc.cluster.local:30000
Secret key:  (leave blank — no credentials required for the local PC-AI S3 service)
```

### Packaging Settings
```
Model name:     e.g. medmnist-cnn-bento-model
Registry:       local-s3-registry
Model format:   Bento archive
Model URL:      s3://mlops-demo/model/medmnist_cnn_demo-bento-v-latest.bento
                  (the exact path is printed by upload_model_to_s3_bucket.py)
Resources:      tiny-cpu template
Env variables:
  BENTO_MODEL_ARTIFACT = medmnist_cnn:<your_tag>   (printed by pick_best.py or bentoml models list)
```

### Deployment Settings
```
Deployment name: medmnist-cnn-bento-model
Packaged model:  medmnist-cnn-bento-model
Scaling:         fixed-1
```

---
## 10. Remote Inference (Deployed MLIS Endpoint)

Once the model is deployed to MLIS (Step 9), call the live endpoint using the access token generated in MLIS.

**Before running the cell below, fill in:**
- `API_TOKEN` — the bearer token generated from the MLIS deployment page
- `MODEL_URI` — the base URL of the deployed endpoint (e.g. `https://<mlis-host>/model`)

In [ ]:
import requests, json

# ── Set your deployed endpoint details ───────────────────────────────────────
API_TOKEN  = ""   # token generated in MLIS
MODEL_URI  = ""   # e.g. https://<mlis-endpoint>/model
INPUT_DATA = "https://drive.google.com/uc?export=download&id=1-jQJ6kvDNEtK_CZaRFicvMDWYz7_qfaf"

url     = f"{MODEL_URI}/predict"
headers = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json",
}
data = {"input_data": INPUT_DATA}

response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)
print(response.json())

---
## Summary

| Step | Description | Tool |
|------|-------------|------|
| 1 | Environment Setup | conda / venv |
| 2 | MLFlow Setup | MLFlow (PC-AI) |
| 3 | Data Preprocessing & Download | MedMNIST |
| 4 | Model Training | PyTorch + Accelerate |
| 5 | Best Model Selection | MLFlow |
| 6 | Inference (Local) | FastAPI / Python |
| 7 | BentoML Packaging | BentoML |
| 8 | Upload to S3 | boto3 / Local S3 |
| 9 | Deploy to MLIS | MLIS |
| 10 | Remote Inference | REST API |

**This pipeline ensures full reproducibility and portability** — whether deploying locally or into a managed ML environment like PC-AI MLIS.